# Looker Studio Address set up

The Building and Properties Violations dataset includes information on all residents in Boston, but our focus is specifically on the violation addresses associated with students. Therefore, we will merge the violation addresses with the student addresses and then visualize the trend in violations from student addresses using Looker Studio.

In [22]:
import pandas as pd

In [23]:
# Read cleaned violations csv file  
df = pd.read_csv('cleaned_violations_data.csv')

In [24]:
df.shape

(7330, 23)

While cleaning the student housing dataset, we set every entry to be in all lowercase. To ensure the merge will work, we will also format our entries to be in all owercase

In [25]:
df['violation_suffix'] = df['violation_suffix'].str.lower()
df['violation_street'] = df['violation_street'].str.lower()

In [26]:
df

,case_no,ap_case_defn_key,status_dttm,status,code,description,violation_stno,violation_stno_range,violation_street,violation_suffix,...,ward,contact_address,contact_company_or_name,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location
0,V740641,1013,2024-05-15,Open,105.1,Failure to Obtain Permit,0,NaN,cedar,st,...,11,142-144 Cedar St,NaN,Roxbury,MA,02119,26750.0,42.327519,-71.094478,"(42.32751857052833, -71.09447759680832)"
1,V391369,1013,2018-04-03,Closed,116.2,Unsafe and Dangerous,1,NaN,allston,st,...,17,1 Allston St,NaN,Dorchester,MA,02124,2713.0,42.291900,-71.066531,"(42.291899536495556, -71.06653061394434)"
2,V619956,1013,2022-07-01,Closed,1010.1.9,Egress Doors,1,3,appleton,st,...,05,250 Dorchester Ave,NaN,South Boston,MA,02127,3815.0,42.346725,-71.069653,"(42.34672453592586, -71.06965349863846)"
3,V691250,1013,2023-09-12,Closed,1001.3.2,Testing & Certification,1,NaN,bay state,pl,...,06,1 Bay State Place,NaN,South Boston,MA,02127,8918.0,42.337879,-71.038110,"(42.33787949463076, -71.03811044681107)"
4,V276778,1013,2016-01-11,Closed,105.1,Failure to Obtain Permit,1,9,beacon,st,...,22,17 Lincoln St,NaN,Newton Highlands,MA,02461,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7325,V419898,1013,2018-10-03,Closed,105.1,Failure to Obtain Permit,992,NaN,dorchester,av,...,13,84 Old Colony Dr,NaN,Weymouth,MA,02188,48817.0,42.316746,-71.056526,"(42.316745521284076, -71.05652553605307)"
7326,V777668,1013,2024-08-08,Open,105.1,Failure to Obtain Permit,993,NaN,hyde park,av,...,18,993 Hyde Park Avenue,NaN,Hyde Park,MA,02136,413789.0,42.265630,-71.121321,"(42.26562961678104, -71.12132079607674)"
7327,V429141,1013,2018-12-11,Closed,105.1,Failure to Obtain Permit,993,997,hyde park,av,...,18,21 Old Farm Road,NaN,Norwood,MA,02062,77720.0,42.265627,-71.121319,"(42.2656266169129, -71.12131879569093)"
7328,V515603,1013,2020-09-15,Closed,4-3,Building or Use of Premise req,995,NaN,massachusetts,av,...,08,2721 N Central Avenue,NaN,Phoenix,AZ,85004,441644.0,42.328262,-71.068882,"(42.32826236799689, -71.06888177514574)"


For the merge, we need to extract the full address for violations. Although we said "full address", we will not include city or state because the student housing dataset does not have these entries. We believe that if we keep these data variables, the merge will not be able to read the addresses to be the same.

In [27]:
# Function to create violation address
def create_address(row):
    stno = row['violation_stno']
    stno_range = row['violation_stno_range']
    
    # Handle NaN or empty range
    if pd.isna(stno_range) or stno_range == '':
        full_stno = f"{stno}"
    else:
        full_stno = f"{stno}-{stno_range}"
    
    # Create the full address with optional street number range
    violations_address = f"{full_stno} {row['violation_street']} {row['violation_suffix']}, {row['violation_zip']}"
    
    # Clean up extra spaces
    return ' '.join(violations_address.split())

# Apply the function to the dataframe
df2 = df.apply(create_address, axis=1)

We tested the new dataframe to see if the format is acceptable on Looker Studio. We were able to map this dataframe, however, we got some datapoints in Europe and Australia on the map. After further research on Google maps, we concluded that we were given the wrong zip code for these datapoints. We will change it to the correct one.

In [28]:
# Correct zip code on two entries (when plotted on looker studio and looking on google maps, zip code wrong 

df2 = df2.replace("2 Roseglen rd, 02126", "2 Roseglen rd, 02136")
df2 = df2.replace("36 Morey rd, 02131", "36 Morey rd, 02132")
df2 = df2.replace("4 Wellington ct, 02119", "4 Wellington ct, 02121")

In [29]:
df2

0                 0 cedar st, 02119
1               1 allston st, 02124
2            1-3 appleton st, 02116
3             1 bay state pl, 02127
4              1-9 beacon st, 02134
                   ...             
7325       992 dorchester av, 02125
7326        993 hyde park av, 02136
7327    993-997 hyde park av, 02136
7328    995 massachusetts av, 02119
7329    995 massachusetts av, 02119
Length: 7330, dtype: object

In [30]:
# Save the dataframe back to a CSV file

output_file_path = "looker_studio_violation_address.csv" 
df2.to_csv(output_file_path, index=False)

We were curious to see which addresses had the most violations occurances:

In [31]:
# Count the occurrences of each address
address_counts = df2.value_counts()

# Show the addresses that occur the most
print("Most frequent addresses:")
print(address_counts[address_counts > 1])

Most frequent addresses:
31 spring garden st, 02125      15
600-610 blue hill av, 02121     14
205-209 humboldt ave, 02121     14
1127-1131 harrison av, 02119    12
13 hendry st, 02122             11
                                ..
115-117 salem st, 02113          2
147 tremont st, 02111            2
956 saratoga st, 02128           2
15 norwell st, 02121             2
821 cummins hwy, 02126           2
Name: count, Length: 1305, dtype: int64


Let us now upload the cleaned student housing dataset for the merge.

In [32]:
df3 = pd.read_csv('final.xlsx - Sheet1.csv')

/var/folders/kz/vb4s2bzd5m59rdxjpt9vyk_h0000gn/T/ipykernel_68775/3701888215.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df3 = pd.read_csv('final.xlsx - Sheet1.csv')


In [33]:
df3

,street_number,street_name,street_suffix,unit_number,zip_code,level_of_study,full_time,extra_large_unit,at_home,year_range,university
0,66,leighton,rd,NaN,02136,u,ft,n,dk,2018-2019,Baptist College
1,66,leighton,rd,NaN,02136,u,ft,n,dk,2018-2019,Baptist College
2,50,rockwell,st,NaN,02124,u,pt,n,dk,2019-2020,Baptist College
3,50,rockwell,st,NaN,02124,u,pt,n,dk,2020-2021,Baptist College
4,75,milton st,west,NaN,02136,u,ft,dk,y,2023-2024,Baptist College
...,...,...,...,...,...,...,...,...,...,...,...
291888,166,foster,street,NaN,2135,g,pt,n,dk,2021-2022,St John Seminary
291889,1105,boylston,street,NaN,2215,g,ft,n,dk,2021-2022,St John Seminary
291890,1105,boylston,street,NaN,2215,g,pt,n,dk,2021-2022,St John Seminary
291891,3,magazine,street,NaN,2119,g,ft,n,dk,2021-2022,St John Seminary


Like the violations dataset, we will extract the full housing address of students

In [34]:
# Function to create housing address

def create_address(row):
    
    # Create the full address with optional street number range
    violations_address = f"{row['street_number']} {row['street_name']} {row['street_suffix']}, {row['zip_code']}"

     # Clean up extra spacecs
    return ' '.join(violations_address.split())

# Apply the function to the dataframe
df4 = df3.apply(create_address, axis=1)

In [35]:
df4

0              66 leighton rd, 02136
1              66 leighton rd, 02136
2              50 rockwell st, 02124
3              50 rockwell st, 02124
4           75 milton st west, 02136
                     ...            
291888       166 foster street, 2135
291889    1105 boylston street, 2215
291890    1105 boylston street, 2215
291891       3 magazine street, 2119
291892       3 magazine street, 2119
Length: 291893, dtype: object

The entries for the street suffix in this dataframe is not consistent. This might be an issue while merging, so we will make the format of the suffix to match the suffix on the violations dataset

In [36]:
# Mapping to match suffixes between datasets

suffix_mapping = {
    'avenue': 'ave',
    'boulevard': 'blvd',
    'circle': 'cir',
    'court': 'ct',
    'drive': 'dr',
    'lane': 'ln',
    'mount': 'mt',
    'parkway': 'pkwy',
    'place': 'pl',
    'road': 'rd',
    'street': 'st',
    'terrace': 'ter',
    'way': 'wy',
    'highway': 'hwy',
    'square': 'sq'
}

# Function to map suffixes
def standardize_address(address):
    address = address.lower()  
    for full_suffix, abbr in suffix_mapping.items():
        if full_suffix in address:
            address = address.replace(full_suffix, abbr)
    return address

# Apply standardization directly to the DataFrame column
df4 = df4.apply(standardize_address)

In [37]:
df4

0            66 leighton rd, 02136
1            66 leighton rd, 02136
2            50 rockwell st, 02124
3            50 rockwell st, 02124
4         75 milton st west, 02136
                    ...           
291888         166 foster st, 2135
291889      1105 boylston st, 2215
291890      1105 boylston st, 2215
291891         3 magazine st, 2119
291892         3 magazine st, 2119
Length: 291893, dtype: object

We noticed that some zip codes in this dataframe includes four extra digits, which we don't have in any of the zip code entries in the violations dataset. To be able to include the entries with the extra digits in the merge, we will remove numbers after the hyphen in the zip code.

In [38]:
df4 = df4.str.replace(r'(\d{5})-\d{4}', r'\1', regex=True)

In [39]:
# Save the dataframe back to a CSV file

output_file_path = "looker_studio_student_address.csv" 
df4.to_csv(output_file_path, index=False)

With both dataframes now having the same format, we can merge the common addresses between the datasets. With this new datadrame, we can map it on Looker Studio to see which student housing address experiences the most violations. 

In [40]:
# New csv file with common addresses between the datasets

common_addresses = df2[df2.isin(df4)]

In [41]:
common_addresses

1          1 allston st, 02124
19      1 devonshire pl, 02108
24         1 fenwick pl, 02119
28      1 gloucester st, 02115
29      1 gloucester st, 02115
                 ...          
7300      98 topliff st, 02122
7301      98 topliff st, 02122
7302      98 topliff st, 02122
7316       99 hudson st, 02111
7324     99 woodrow ave, 02124
Length: 1608, dtype: object

In [42]:
# Save the dataframe back to a CSV file

output_file_path = "looker_studio_common_address.csv" 
common_addresses.to_csv(output_file_path, index=False)

In [43]:
# Count the occurrences of each address
address_counts2 = common_addresses.value_counts()

# Show the addresses that occur the most
print("Most frequent addresses:")
print(address_counts2[address_counts2 > 1])

Most frequent addresses:
1 rosa st, 02136             9
47 cedar st, 02114           7
1654 washington st, 02118    7
37 beacon st, 02108          7
169 beacon st, 02116         6
                            ..
76 brookline st, 02118       2
9 wallingford rd, 02135      2
317 wood ave, 02136          2
11 marlborough st, 02116     2
74 corey rd, 02135           2
Name: count, Length: 297, dtype: int64


# Sam-ID Linking Exploratory Analysis 

In [44]:
# Turn df2 into a dataframe

df5 = pd.DataFrame(df2).copy()
df5.columns = ["violations_address"]

In [45]:
df5

,violations_address
0,"0 cedar st, 02119"
1,"1 allston st, 02124"
2,"1-3 appleton st, 02116"
3,"1 bay state pl, 02127"
4,"1-9 beacon st, 02134"
...,...
7325,"992 dorchester av, 02125"
7326,"993 hyde park av, 02136"
7327,"993-997 hyde park av, 02136"
7328,"995 massachusetts av, 02119"


In [46]:
df6 = df.join(df5)

In [47]:
df6.to_csv('Looker_Studio_with_Samid.csv')

In [48]:
df6.head()

,case_no,ap_case_defn_key,status_dttm,status,code,description,violation_stno,violation_stno_range,violation_street,violation_suffix,...,contact_address,contact_company_or_name,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location,violations_address
0,V740641,1013,2024-05-15,Open,105.1,Failure to Obtain Permit,0,NaN,cedar,st,...,142-144 Cedar St,NaN,Roxbury,MA,02119,26750.0,42.327519,-71.094478,"(42.32751857052833, -71.09447759680832)","0 cedar st, 02119"
1,V391369,1013,2018-04-03,Closed,116.2,Unsafe and Dangerous,1,NaN,allston,st,...,1 Allston St,NaN,Dorchester,MA,02124,2713.0,42.291900,-71.066531,"(42.291899536495556, -71.06653061394434)","1 allston st, 02124"
2,V619956,1013,2022-07-01,Closed,1010.1.9,Egress Doors,1,3,appleton,st,...,250 Dorchester Ave,NaN,South Boston,MA,02127,3815.0,42.346725,-71.069653,"(42.34672453592586, -71.06965349863846)","1-3 appleton st, 02116"
3,V691250,1013,2023-09-12,Closed,1001.3.2,Testing & Certification,1,NaN,bay state,pl,...,1 Bay State Place,NaN,South Boston,MA,02127,8918.0,42.337879,-71.038110,"(42.33787949463076, -71.03811044681107)","1 bay state pl, 02127"
4,V276778,1013,2016-01-11,Closed,105.1,Failure to Obtain Permit,1,9,beacon,st,...,17 Lincoln St,NaN,Newton Highlands,MA,02461,NaN,NaN,NaN,NaN,"1-9 beacon st, 02134"


In [49]:
df6[df6["sam_id"] == 129629.0]

,case_no,ap_case_defn_key,status_dttm,status,code,description,violation_stno,violation_stno_range,violation_street,violation_suffix,...,contact_address,contact_company_or_name,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location,violations_address
3691,V279276,1013,2016-01-22,Closed,116.2,Unsafe and Dangerous,31,NaN,spring garden,st,...,14 Whiting Rd,C/O Jeff Abrams,Dover,MA,02030-2451,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3692,V313345,1013,2017-01-17,Closed,105.1,Failure to Obtain Permit,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3693,V268901,1013,2016-09-21,Closed,116.2,Unsafe and Dangerous,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3694,V306981,1013,2016-09-01,Closed,1015.1,Exits,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3695,V313390,1013,2016-08-16,Closed,115,Stop Work Order,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3696,V306984,1013,2016-08-04,Closed,701.1,Fire and Smoke Protection,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3697,V274757,1013,2016-04-04,Closed,12.6.1.1,Metal Chimeys,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3698,V274757,1013,2016-04-04,Closed,12.4.2,Design/construction of vent.,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3699,V292558,1013,2016-03-29,Closed,1006.3,Means of Egress,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3700,V292557,1013,2016-03-29,Closed,105.1,Failure to Obtain Permit,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"


In [50]:
df6["sam_id"].value_counts()

sam_id
0.0         19
129629.0    15
75983.0     14
162187.0    14
166058.0    12
            ..
67171.0      1
71433.0      1
75646.0      1
80395.0      1
44093.0      1
Name: count, Length: 5109, dtype: int64

In [51]:
grouped = df6.groupby(["sam_id", "violations_address"]).size()
group_new = grouped.reset_index(name = "violation_count")

In [52]:
group_new.to_csv('Looker_Studio_with_Samid_and_Count.csv')

In [53]:
df6[df6["violations_address"] == "31 spring garden st, 02125"]

,case_no,ap_case_defn_key,status_dttm,status,code,description,violation_stno,violation_stno_range,violation_street,violation_suffix,...,contact_address,contact_company_or_name,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location,violations_address
3691,V279276,1013,2016-01-22,Closed,116.2,Unsafe and Dangerous,31,NaN,spring garden,st,...,14 Whiting Rd,C/O Jeff Abrams,Dover,MA,02030-2451,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3692,V313345,1013,2017-01-17,Closed,105.1,Failure to Obtain Permit,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3693,V268901,1013,2016-09-21,Closed,116.2,Unsafe and Dangerous,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3694,V306981,1013,2016-09-01,Closed,1015.1,Exits,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3695,V313390,1013,2016-08-16,Closed,115,Stop Work Order,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3696,V306984,1013,2016-08-04,Closed,701.1,Fire and Smoke Protection,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3697,V274757,1013,2016-04-04,Closed,12.6.1.1,Metal Chimeys,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3698,V274757,1013,2016-04-04,Closed,12.4.2,Design/construction of vent.,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3699,V292558,1013,2016-03-29,Closed,1006.3,Means of Egress,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"
3700,V292557,1013,2016-03-29,Closed,105.1,Failure to Obtain Permit,31,NaN,spring garden,st,...,14 Whiting Rd,NaN,Dover,MA,02030,129629.0,42.31748,-71.053511,"(42.31747951681026, -71.05351052720917)","31 spring garden st, 02125"


In [54]:
grouped = group_new.groupby(["sam_id", "violation_count"]).size()
group_new = grouped.reset_index(name = "total_violation_count")

In [55]:
df6["violations_address"].value_counts()

violations_address
31 spring garden st, 02125      15
600-610 blue hill av, 02121     14
205-209 humboldt ave, 02121     14
1127-1131 harrison av, 02119    12
13 hendry st, 02122             11
                                ..
23 glencoe st, 02135             1
23 goethe st, 02132              1
23 grayson st, 02124             1
23 grew hill rd, 02131           1
309R- sumner st, 02128           1
Name: count, Length: 5136, dtype: int64